In [6]:
from datetime import datetime, timedelta
import pandas as pd

# Define the base year-independent holiday structure
holidays_by_year = {
    "01.01": "Alle Bundesländer",  # New Year's Day (fixed date)
    "06.01": "Baden-Wuerttemberg, Bayern, Sachsen-Anhalt",  # Epiphany (fixed date)
    "01.05": "Alle Bundesländer",  # Labor Day (fixed date)
    "03.10": "Alle Bundesländer",  # German Unity Day (fixed date)
    "31.10": "Brandenburg, Bremen, Hamburg, Mecklenburg-Vorpommern, Niedersachsen, Sachsen, Sachsen-Anhalt, Schleswig-Holstein, Thueringen",  # Reformation Day (fixed date)
    "01.11": "Baden-Wuerttemberg, Bayern, Nordrhein-Westfalen, Rheinland-Pfalz, Saarland",  # All Saints' Day (fixed date)
    "25.12": "Alle Bundesländer",  # Christmas Day (fixed date)
    "26.12": "Alle Bundesländer",  # Boxing Day (fixed date)
}

# Define movable holidays based on Easter Sunday
movable_holidays_offsets = {
    "Karfreitag": -2,  # Good Friday (2 days before Easter)
    "Ostermontag": 1,  # Easter Monday (1 day after Easter)
    "Christi Himmelfahrt": 39,  # Ascension Day (39 days after Easter)
    "Pfingstmontag": 50,  # Whit Monday (50 days after Easter)
    "Fronleichnam": 60,  # Corpus Christi (60 days after Easter)
}

# Additional state-specific holidays with fixed logic
state_specific_holidays = {
    "15.08": "Bayern (Catholic areas), Saarland",  # Assumption Day
    "20.11": "Sachsen",  # Repentance and Prayer Day (Wednesday before last Sunday of church year)
}

tso_mapping = {
    "Nordrhein-Westfalen":"Ampiron", "Saarland":"Ampiron", "Rheinland-Pfalz":"Ampiron", "Baden-Wuerttemberg":"TransnetBW", "Niedersachsen":"Tennet",
    "Schleswig-Holstein":"Tennet", "Bremen":"Tennet", "Bayern":"Tennet", "Hessen":"Tennet", "Brandenburg":"50Hertz", "Berlin":"50Hertz",
    "Hamburg":"50Hertz","Sachsen-Anhalt":"50Hertz", "Mecklenburg-Vorpommern":"50Hertz", "Thueringen":"50Hertz", "Sachsen":"50Hertz"
}

# Function to calculate Easter Sunday for a given year (Gauss' Easter Algorithm)
def calculate_easter_sunday(year):
    a = year % 19
    b = year // 100
    c = year % 100
    d = b // 4
    e = b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i = c // 4
    k = c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = ((h + l - 7 * m + 114) % 31) + 1
    return datetime(year, month, day)

In [7]:

# Generate holidays for each year
all_data = []
for year in range(2020, 2028):
    # Fixed holidays
    for date, bundeslaender in holidays_by_year.items():
        day, month = map(int, date.split("."))
        holiday_date = datetime(year, month, day).strftime("%d.%m.%Y")
        all_data.append([holiday_date, bundeslaender])
    
    # Easter-based holidays
    easter_sunday = calculate_easter_sunday(year)
    for name, offset in movable_holidays_offsets.items():
        holiday_date = (easter_sunday + timedelta(days=offset)).strftime("%d.%m.%Y")
        bundeslaender = "Alle Bundesländer" if name != "Fronleichnam" else "Baden-Wuerttemberg, Bayern, Hessen, Nordrhein-Westfalen, Rheinland-Pfalz, Saarland"
        all_data.append([holiday_date, bundeslaender])
    
    # State-specific fixed holidays
    for date, bundeslaender in state_specific_holidays.items():
        day, month = map(int, date.split("."))
        holiday_date = datetime(year, month, day).strftime("%d.%m.%Y")
        all_data.append([holiday_date, bundeslaender])

# Convert holiday data to DataFrame and add TSO columns
final_data = []
for date, bundeslaender in all_data:
    tso_status = {"Amprion": 0, "TenneT DE": 0, "TransnetBW": 0, "50Hertz": 0}

    if bundeslaender == "Alle Bundesländer":
        tso_status = {key: 1 for key in tso_status}
    else:
        for land in bundeslaender.split(", "):
            land = land.split(" (")[0]  # Remove any additional text in parentheses

            if land in tso_mapping:
                tso_status[tso_mapping[land]] = 1

    final_data.append([date, bundeslaender, tso_status["Amprion"], tso_status["TenneT DE"], tso_status["TransnetBW"], tso_status["50Hertz"]])

# Convert to DataFrame and export to CSV format
df_final = pd.DataFrame(final_data, columns=["day", "Bundesländer", "Amprion", "TenneT DE", "TransnetBW", "50Hertz"])

In [9]:
all_days = pd.date_range(start="2020-01-01", end="2027-12-31", freq="d")

In [13]:
df_final["day"] = pd.to_datetime(df_final["day"], format="%d.%m.%Y")

In [19]:
df_final.to_csv("../data/holidays_new.csv", index=False)